# 00 · Start here: the problem, the idea, and LangGraph from scratch

**You'll learn:** why fraud detection needs more than rules or "just ask an LLM", the tiered design this project uses, and the LangGraph building blocks it is made of: state, nodes, edges, routing, parallel branches, pausing for a human, and memory. Each block gets a tiny toy example.

**Time:** about 15 minutes · **Cost:** free · **Needs:** nothing beyond the installed project. No API key, no LLM calls.

| Part | What it covers |
|---|---|
| 1 | The problem |
| 2 | The idea: tiered escalation |
| 3 | Setup |
| 4 | LangGraph building blocks, hands-on: 4.1 state, nodes, edges · 4.2 routing · 4.3 parallel branches · 4.4 pause for a human · 4.5 memory across runs |
| 5 | Where to go next |

---
## Part 1 · The problem

A payments company sees millions of transactions. A few are fraud. Two obvious approaches both fail:

| Approach | Strength | What goes wrong |
|---|---|---|
| **Rules only** ("block anything over $5,000 abroad") | instant, free, predictable | rigid: misses fraud that doesn't break a rule, and blocks honest customers who do |
| **An LLM investigates every transaction** | reasons about context ("is a new device suspicious *for this account*?") | slow and expensive: 3–4 LLM calls on each of millions of transactions, when 95%+ are obvious |

And for the truly hard cases there is a third requirement: **someone accountable decides.** Nobody wants an LLM alone freezing a customer's $18,000 wire transfer.

---
## Part 2 · The idea: tiered escalation

Spend effort only where it is needed:

1. **Triage** with cheap rules: obvious cases are approved or declined instantly, with no LLM.
2. **Investigate** only the ambiguous middle: three AI specialists look at it in parallel, each from a different angle.
3. **Aggregate** their findings into one risk score and explanation.
4. **Ask a human** when risk stays high. The pipeline pauses, saves its state to disk, and waits.
5. **Resolve** and remember the decision for next time.

```mermaid
flowchart LR
  I["ingest"] --> T{"triage<br/>rules, no LLM"}
  T -->|"score < 0.30"| AA["auto_approve"]
  T -->|"score ≥ 0.65"| AD["auto_decline"]
  T -->|"in between"| TA["transaction_analyst"]
  T -->|"in between"| IV["identity_verifier"]
  T -->|"in between"| PD["pattern_detector"]
  TA --> AG["aggregator"]
  IV --> AG
  PD --> AG
  AG -->|"composite ≥ 0.5"| HR["human_review<br/>pauses here"]
  AG -->|"below 0.5"| R["resolution"]
  HR --> R
  AA --> R
  AD --> R
  R --> E(["END"])
```

Every box is a **node** in a LangGraph graph. Part 4 builds each mechanism on toy examples first.

---
## Part 3 · Setup

### Setup
The next cell connects the notebook to the project. It gives this notebook **its own** database files in `.notebook_state/`, so nothing here touches the Streamlit app's data. It prints which LLM this run uses and whether its key is set (never the key itself). Set `SHOW_LIVE_LOGS = True` to see each node's live log line.

In [1]:
# ── Settings ────────────────────────────────────────────────────────────
SHOW_LIVE_LOGS = False  # True = print every node's live log line, like the terminal does

# Setup: run from the repo root, and give this notebook its own database files in
# .notebook_state/ so it never touches the app's checkpoints.sqlite / account_memory.sqlite.
import logging
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
STATE_DIR = ROOT / ".notebook_state"
STATE_DIR.mkdir(exist_ok=True)
os.environ["CHECKPOINT_DB_PATH"] = str(STATE_DIR / "00_checkpoints.sqlite")
os.environ["MEMORY_DB_PATH"] = str(STATE_DIR / "00_memory.sqlite")
os.environ.setdefault("FASTMCP_LOG_LEVEL", "WARNING")  # quiet the MCP server subprocess
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")  # never overrides the two database paths set above

import pandas as pd

import core.audit  # noqa: F401  (creates the "fraud_guard" live logger)
from core.llm import llm_model, llm_provider

logging.getLogger("fraud_guard").setLevel(logging.INFO if SHOW_LIVE_LOGS else logging.WARNING)
pd.set_option("display.max_colwidth", 160)
key = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY"}.get(llm_provider(), "?")
print(f"LLM: {llm_provider()} / {llm_model()}   ({key} set: {'yes' if os.environ.get(key) else 'NO'})")

LLM: gemini / gemini-3.5-flash   (GEMINI_API_KEY set: yes)


---
## Part 4 · LangGraph building blocks, hands-on

### 4.1 State, nodes and edges

A LangGraph app is a **graph**:
- **State**: one shared dictionary (here a `TypedDict`) that flows through the graph.
- **Node**: a plain Python function. It receives the state and returns **only the keys it changes**.
- **Edge**: "after this node, run that one". `START` and `END` mark the entry and exit.

`compile()` turns the description into something you can `invoke()`.

In [2]:
from typing import TypedDict

from IPython.display import Markdown
from langgraph.graph import END, START, StateGraph


def show(app):
    """Draw a compiled graph (JupyterLab renders the Mermaid diagram)."""
    return Markdown("```mermaid\n" + app.get_graph().draw_mermaid() + "\n```")


class ToyState(TypedDict):
    amount: float
    note: str


def check_amount(state: ToyState) -> dict:
    return {"note": "small purchase" if state["amount"] < 100 else "large purchase"}


builder = StateGraph(ToyState)
builder.add_node("check_amount", check_amount)
builder.add_edge(START, "check_amount")
builder.add_edge("check_amount", END)
toy = builder.compile()

print(toy.invoke({"amount": 6.50, "note": ""}))
show(toy)

{'amount': 6.5, 'note': 'small purchase'}


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	check_amount(check_amount)
	__end__([<p>__end__</p>]):::last
	__start__ --> check_amount;
	check_amount --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

**What to notice:** the node returned only `note`, and LangGraph merged it into the state.

### 4.2 Routing with a conditional edge

A **conditional edge** calls a small *router* function that looks at the state and returns the name of the next node. In Fraud Guard, `route_after_triage()` picks approve, decline or investigate, and `route_after_aggregator()` picks human review or resolution.

In [3]:
class TxnState(TypedDict):
    amount: float
    decision: str


def score(state: TxnState) -> dict:
    return {}


def route(state: TxnState) -> str:
    return "approve" if state["amount"] < 100 else "review"


builder = StateGraph(TxnState)
builder.add_node("score", score)
builder.add_node("approve", lambda s: {"decision": "approved"})
builder.add_node("review", lambda s: {"decision": "sent for review"})
builder.add_edge(START, "score")
builder.add_conditional_edges("score", route, {"approve": "approve", "review": "review"})
builder.add_edge("approve", END)
builder.add_edge("review", END)
router_app = builder.compile()

for amount in (6.50, 18000):
    print(amount, "->", router_app.invoke({"amount": amount, "decision": ""})["decision"])
show(router_app)

6.5 -> approved
18000 -> sent for review


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	score(score)
	approve(approve)
	review(review)
	__end__([<p>__end__</p>]):::last
	__start__ --> score;
	score -.-> approve;
	score -.-> review;
	approve --> __end__;
	review --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

### 4.3 Parallel branches and reducers

When one node has edges to **several** nodes, they run **in parallel** (fan-out); edges from all of them into one node wait for every branch (fan-in). That is how the three specialists run at the same time.

Parallel nodes writing to the **same** key need a rule for combining their writes: a **reducer**. `Annotated[list, operator.add]` means "append". Fraud Guard's `audit_log` works exactly like this. Without a reducer, LangGraph refuses the conflicting writes, as the second cell shows.

In [4]:
import operator
from typing import Annotated


class InvestigationState(TypedDict):
    findings: Annotated[list[str], operator.add]  # reducer: parallel writes are appended


def build_investigation(state_type):
    builder = StateGraph(state_type)
    builder.add_node("start", lambda s: {})
    builder.add_node("analyst", lambda s: {"findings": ["analyst: amount is 4x the usual"]})
    builder.add_node("identity", lambda s: {"findings": ["identity: new device"]})
    builder.add_node("patterns", lambda s: {"findings": ["patterns: resembles a known scam"]})
    builder.add_node("combine", lambda s: {"findings": [f"combined {len(s['findings'])} findings"]})
    builder.add_edge(START, "start")
    for name in ("analyst", "identity", "patterns"):
        builder.add_edge("start", name)   # fan-out: all three run in parallel
        builder.add_edge(name, "combine")  # fan-in: combine waits for all three
    builder.add_edge("combine", END)
    return builder.compile()


parallel_app = build_investigation(InvestigationState)
for line in parallel_app.invoke({"findings": []})["findings"]:
    print(line)
show(parallel_app)

analyst: amount is 4x the usual
identity: new device
patterns: resembles a known scam
combined 3 findings


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	start(start)
	analyst(analyst)
	identity(identity)
	patterns(patterns)
	combine(combine)
	__end__([<p>__end__</p>]):::last
	__start__ --> start;
	analyst --> combine;
	identity --> combine;
	patterns --> combine;
	start --> analyst;
	start --> identity;
	start --> patterns;
	combine --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [5]:
class NoReducerState(TypedDict):
    findings: list[str]  # no reducer


try:
    build_investigation(NoReducerState).invoke({"findings": []})
except Exception as exc:
    print(type(exc).__name__, "->", str(exc)[:160])

InvalidUpdateError -> At key 'findings': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/o


### 4.4 Pausing for a human: interrupts and checkpoints

Two features work together:
- A **checkpointer** saves the state after every step, keyed by a `thread_id` (one per run).
- `interrupt_before=["human_review"]` stops the graph **before** that node runs.

The run then waits. Later (seconds or days, even after a restart when the checkpointer writes to disk), you add the human's decision with `update_state()` and continue with `invoke(None, config)`. Fraud Guard uses a SQLite checkpointer; this toy uses an in-memory one.

In [6]:
from langgraph.checkpoint.memory import InMemorySaver


class ReviewState(TypedDict):
    amount: float
    human_decision: str
    final: str


builder = StateGraph(ReviewState)
builder.add_node("triage", lambda s: {})
builder.add_node("human_review", lambda s: {})
builder.add_node("resolve", lambda s: {"final": f"{s['human_decision']}d by a human"})
builder.add_edge(START, "triage")
builder.add_edge("triage", "human_review")
builder.add_edge("human_review", "resolve")
builder.add_edge("resolve", END)
review_app = builder.compile(checkpointer=InMemorySaver(), interrupt_before=["human_review"])

config = {"configurable": {"thread_id": "toy-wire-1"}}
review_app.invoke({"amount": 18000, "human_decision": "", "final": ""}, config)
print("paused before:", review_app.get_state(config).next)

review_app.update_state(config, {"human_decision": "approve"})  # the reviewer clicks Approve
print("after resume:", review_app.invoke(None, config)["final"])
show(review_app)

paused before: ('human_review',)
after resume: approved by a human


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	triage(triage)
	human_review(human_review<hr/><small><em>__interrupt = before</em></small>)
	resolve(resolve)
	__end__([<p>__end__</p>]):::last
	__start__ --> triage;
	human_review --> resolve;
	triage --> human_review;
	resolve --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

### 4.5 Memory across runs: the store

The checkpointer remembers **one run**. A **store** remembers things **across runs**, organised by namespace, e.g. `("account_memory", "acct_2002")`. Nodes read it with `get_store()`. Fraud Guard's `resolution` node writes each decision; next time, `triage` reads the account's history and raises the risk score if it was escalated or declined before.

In [7]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(("account_memory", "acct_2002"), "txn_003", {"final_decision": "escalated"})
store.put(("account_memory", "acct_2002"), "txn_004", {"final_decision": "approved"})
[item.value for item in store.search(("account_memory", "acct_2002"))]

[{'final_decision': 'escalated'}, {'final_decision': 'approved'}]

---
## Part 5 · Where to go next

| Notebook | What you'll do | API key? |
|---|---|---|
| **01_triage_rules** | The rule engine: signals, scores, thresholds, routing, the memory penalty, and how much traffic reaches the LLM | no |
| **02_specialist_agents** | Tools, the fraud-case search, the live MCP geolocation server, ReAct agents, the three specialists and the aggregator | yes |
| **03_full_graph** | The whole graph on real scenarios: auto paths, the pause for review, the human decision, the audit trail | yes |
| **04_persistence_and_memory** | Surviving a restart mid-review, checkpoint history, long-term account memory, Studio, CLI and app | yes |

Each notebook runs on its own.